# 🔧 Pronóstico Integrado v3 ROBUSTO — Oil Separator (Basic No. 1055)
## VW Tiguan L 2.0T Motor DTEA · Planta Puebla · MY 2021–2024
### Versión 3: 6 mejoras adicionales sobre v2 (11 total)

---

**Mejoras acumuladas:**

| # | Mejora | Versión | Beneficio |
|---|--------|---------|----------|
| 1 | Pesos ensamble por MAPE de CV | v2 | Elimina sesgo por overfitting en pesos |
| 2 | Bootstrap residual para IC genuinos | v2 | IC 90% empírico, no pragmático |
| 3 | Backtest extendido 24 meses | v2 | Mide precisión real a 2 años |
| 4 | Ridge regression regularizada | v2 | Generaliza mejor con poca data |
| 5 | Monte Carlo del offset prod→venta | v2 | Propaga incertidumbre estructural |
| **6** | **Weibull MLE censurado** | **v3** | **Corrige sesgo β y η para MY2023/24 (100% en garantía)** |
| **7** | **Feature at-risk population** | **v3** | **Física de flota cerrada codificada en ML** |
| **8** | **Reducción features: 33→22** | **v3** | **Ratio n/p: 0.82→1.23; elimina overfitting estructural** |
| **9** | **Random Forest + ElasticNet** | **v3** | **Modelos robustos para n pequeño; OOB validation** |
| **10** | **Meta-learner stacking** | **v3** | **Pesos basados en OOF predictions reales** |
| **11** | **Transición Weibull anticipada** | **v3** | **Flota cerrada: Weibull domina desde mes 7** |

---
**Supuestos:** Flota cerrada 435,801 unidades · Garantía 48 meses desde venta · Inflación 3% anual  
**Fin producción DTEA:** Semana 46/2024 (≈ Noviembre 2024)

## 1. Librerías y Configuración

In [ ]:
import warnings, os
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
import matplotlib.patches as mpatches
import scipy.stats as stats
from scipy.optimize import minimize
from itertools import product

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import RidgeCV, ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error
from xgboost import XGBRegressor

PALETTE = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b','#e377c2','#17becf']
plt.rcParams.update({'figure.dpi':120,'axes.spines.top':False,'axes.spines.right':False,
                     'axes.grid':True,'grid.alpha':0.3,'font.size':10})

DATA_FILE        = 'Info_Juan.xlsx'
VOL_FILE         = '2021_No_of_vehicles_by_production_month_total_VEH.xlsx'
WARR_MONTHS      = 48
MY_LIST          = [2021, 2022, 2023, 2024]
HORIZON          = 120
HORIZON_END      = pd.Period('2036-04', freq='M')
INFLATION        = 0.03
N_LAGS           = 6          # v3: reducido de 12
STABLE_START     = 13
XGB_SHORT        = 6          # v3: reducido de 12 (flota cerrada)
XGB_BLEND        = 12         # v3: reducido de 24
N_BOOTSTRAP      = 1000
N_MC             = 500
PROD_CUTOFF      = pd.Period('2024-11', freq='M')  # Semana 46/2024
SARIMA_THRESHOLD = 15.0       # excluir SARIMA si CV-MAPE > 15%

print('✅ Configuración v3 cargada')
print(f'   PROD_CUTOFF     : {PROD_CUTOFF} (semana 46/2024 — flota cerrada)')
print(f'   N_LAGS          : {N_LAGS} (reducido de 12)')
print(f'   XGB_SHORT/BLEND : {XGB_SHORT}/{XGB_BLEND} meses (reducido de 12/24)')
print(f'   SARIMA_THRESHOLD: >{SARIMA_THRESHOLD}% CV-MAPE → excluir del ensamble')

## 2. Carga de Datos y Análisis del Offset Producción→Venta

In [ ]:
df = pd.read_excel(DATA_FILE)
df['Repair date']     = pd.to_datetime(df['Repair date'],     errors='coerce')
df['SALES DATE']      = pd.to_datetime(df['SALES DATE'],      errors='coerce')
df['Production Date'] = pd.to_datetime(df['Production Date'], errors='coerce')
df = df[df['Repair date'] < '2026-05-01'].copy()
for c in ['Material cost','Labour cost','COSTS']:
    df[c] = df[c].clip(lower=0)
df['en_garantia'] = df['MIS'] <= WARR_MONTHS
df['cal_month']   = df['Repair date'].dt.to_period('M')

df_vol = pd.read_excel(VOL_FILE, usecols=[0,1])
df_vol.columns = ['prod_month','n_prod']
df_vol['prod_period'] = pd.to_datetime(df_vol['prod_month']).dt.to_period('M')
df_vol = df_vol[df_vol['n_prod'] > 0][['prod_period','n_prod']].copy()
df_vol['model_year'] = df_vol['prod_period'].dt.year.clip(2021, 2024)
df_vol.loc[df_vol['prod_period'].dt.year == 2020, 'model_year'] = 2021

# Distribución empírica del offset producción→venta (Mejora 5)
df_off = df.dropna(subset=['SALES DATE','Production Date']).copy()
df_off['offset_real'] = ((df_off['SALES DATE'].dt.to_period('M') -
                          df_off['Production Date'].dt.to_period('M'))
                         .apply(lambda x: x.n if hasattr(x,'n') else np.nan))
df_off = df_off[df_off['offset_real'].between(0,18)]
OFFSET_DIST   = df_off['offset_real'].values.astype(int)
OFFSET_MEDIAN = int(np.median(OFFSET_DIST))

df_vol['sales_period_est'] = df_vol['prod_period'] + OFFSET_MEDIAN
df_vol['warr_end']         = df_vol['sales_period_est'] + WARR_MONTHS

print(f'Registros fallas    : {len(df):,}')
print(f'Parque producido    : {df_vol["n_prod"].sum():,} unidades')
print(f'Offset mediana      : {OFFSET_MEDIAN} meses  (media={OFFSET_DIST.mean():.2f}, std={OFFSET_DIST.std():.2f})')
print(f'Última cohorte prod : {df_vol["prod_period"].max()}')
print(f'Última venta est.   : {df_vol["sales_period_est"].max()}')
print(f'Última garantía     : {df_vol["warr_end"].max()}')

## 3. Series Históricas Mensuales

In [ ]:
full_idx   = pd.period_range('2021-12','2026-04', freq='M')
hist_w = df[df['en_garantia']].groupby('cal_month').size().reindex(full_idx).fillna(0)
hist_p = df[~df['en_garantia']].groupby('cal_month').size().reindex(full_idx).fillna(0)
hist_t = hist_w + hist_p
hist_dates = full_idx.to_timestamp()

ts_w = hist_w.values.astype(float)
ts_p = hist_p.values.astype(float)
post_ratio = hist_p.sum() / hist_w.sum()

ts_w_stable = ts_w[STABLE_START:]
ts_p_stable = ts_p[STABLE_START:]

cost_w = df[df['en_garantia']].groupby('cal_month').agg(
    mat=('Material cost','sum'), lab=('Labour cost','sum'), total=('COSTS','sum')
).reindex(full_idx).fillna(0)
cost_p = df[~df['en_garantia']].groupby('cal_month').agg(
    mat=('Material cost','sum'), lab=('Labour cost','sum'), total=('COSTS','sum')
).reindex(full_idx).fillna(0)

park_by_my  = df_vol.groupby('model_year')['n_prod'].sum()
fails_by_my = df.groupby('Model Year').size()

print(f'Serie completa  : {len(ts_w)} meses (Dic-2021 → Abr-2026)')
print(f'Serie estable   : {len(ts_w_stable)} meses (Ene-2023 → Abr-2026)')
print(f'Media garantía  : {ts_w_stable.mean():.1f}/mes')
print(f'Media sin gar.  : {ts_p_stable.mean():.1f}/mes')
print(f'Ratio pos/warr  : {post_ratio*100:.2f}%')

## 4. Mejora 6: Weibull MLE con Censura Correcta

**Origen de la censura:** La garantía dura exactamente `WARR_MONTHS=48` meses desde la fecha de venta.
Los vehículos sin falla tienen tiempo de censura `c = min(MIS_actual, 48)`.

| MY | MIS en Abr-2026 | % en garantía | Impacto censura |
|----|----------------|--------------|----------------|
| 2021 | 60-64m | 0% | Mínimo (censura en 48m) |
| 2022 | 38-49m | ~20% | Pequeño |
| 2023 | 26-37m | **100%** | **Alto** |
| 2024 | 15-25m | **100%** | **Muy alto** |

**Por qué `fail_pen` no basta:** Corrige el NIVEL total pero no la FORMA temporal (β).
Si β está sesgado, el pico de fallas se predice en el momento equivocado.

**Log-likelihood correcto:**
`log-L = Σ log[f(tᵢ; β,η)] + Σ nⱼ · log[S(cⱼ; β,η)]`
donde `S = 1-F` es la función de supervivencia.

In [ ]:
def fit_weibull_censored(failures, censored_dict):
    """
    MLE para Weibull con datos censurados por la derecha.
    failures      : array MIS de fallas observadas (dentro de garantia)
    censored_dict : {t_censura: n_vehiculos} para vehículos sin falla
    """
    failures = np.asarray(failures, dtype=float)
    failures = failures[(failures > 0) & (failures <= WARR_MONTHS)]
    if len(failures) < 10:
        sh, _, sc = stats.weibull_min.fit(failures, floc=0)
        return float(sh), float(sc)

    def neg_loglik(log_params):
        beta = np.exp(log_params[0])
        eta  = np.exp(log_params[1])
        ll = np.sum(stats.weibull_min.logpdf(failures, beta, 0, eta))
        for t_c, n_c in censored_dict.items():
            if t_c > 0 and n_c > 0:
                ll += n_c * stats.weibull_min.logsf(float(t_c), beta, 0, eta)
        return -ll if np.isfinite(ll) else 1e12

    sh0, _, sc0 = stats.weibull_min.fit(failures, floc=0)
    x0 = [np.log(max(sh0, 0.5)), np.log(max(sc0, 1.0))]
    res_nm   = minimize(neg_loglik, x0, method='Nelder-Mead',
                        options={'maxiter':10000,'xatol':1e-8,'fatol':1e-8})
    res_bfgs = minimize(neg_loglik, x0, method='L-BFGS-B',
                        bounds=[(-0.5, 4), (1, 6)],
                        options={'maxiter':2000,'ftol':1e-10})
    best = res_bfgs if (np.isfinite(res_bfgs.fun) and res_bfgs.fun < res_nm.fun) else res_nm
    return float(np.exp(best.x[0])), float(np.exp(best.x[1]))


ref_period = pd.Period('2026-04', freq='M')
wb_params  = {}
fail_pen   = {}

print('Fitting Weibull MLE censurado por Model Year (Mejora 6)...\n')
print(f'{"MY":<6} {"N_fail":>8} {"N_park":>10} {"N_cens":>10} '
      f'{"β_prev":>8} {"β_corr":>8} {"η_prev":>8} {"η_corr":>8} {"B50":>7} {"Δη%":>7}')
print('─' * 95)

for my in MY_LIST:
    mis_fail = df[(df['Model Year']==my) & (df['en_garantia'])]['MIS'].dropna().values
    n_park   = int(park_by_my.get(my, 0))
    n_fails  = len(mis_fail)
    n_cens   = max(n_park - n_fails, 0)

    sub = df_vol[df_vol['model_year']==my].copy()
    sub['current_mis'] = sub['sales_period_est'].apply(
        lambda s: max((ref_period - s).n, 0))
    sub['cens_time'] = np.minimum(sub['current_mis'].values, WARR_MONTHS)
    total_prod = sub['n_prod'].sum()
    sub['n_cens_cohort'] = (sub['n_prod'] / total_prod * n_cens
                            ).round().clip(lower=0).astype(int)
    cens_dict = sub.groupby('cens_time')['n_cens_cohort'].sum().to_dict()

    sh_prev, _, sc_prev = stats.weibull_min.fit(mis_fail, floc=0)
    sh_corr, sc_corr   = fit_weibull_censored(mis_fail, cens_dict)
    b50 = stats.weibull_min.ppf(0.50, sh_corr, 0, sc_corr)
    delta_eta_pct = (sc_corr - sc_prev) / sc_prev * 100

    sub2 = sub.copy()
    sub2['mis_r'] = sub2['sales_period_est'].apply(
        lambda s: max((ref_period - s).n, 0))
    sub2['p_wb'] = sub2['mis_r'].apply(
        lambda t: stats.weibull_min.cdf(min(t, WARR_MONTHS), sh_corr, 0, sc_corr))
    p_avg = (sub2['p_wb'] * sub2['n_prod']).sum() / sub2['n_prod'].sum()
    pen   = n_fails / (n_park * p_avg) if p_avg > 0.001 else 0.0

    wb_params[my] = {'sh': sh_corr, 'sc': sc_corr, 'n': n_fails, 'B50': b50}
    fail_pen[my]  = pen

    print(f'MY{my}  {n_fails:>8,} {n_park:>10,} {n_cens:>10,} '
          f'{sh_prev:>8.3f} {sh_corr:>8.3f} {sc_prev:>8.1f} {sc_corr:>8.1f} '
          f'{b50:>7.1f} {delta_eta_pct:>+7.1f}%')

print(f'\nPenetraciones (parámetros corregidos):')
for my in MY_LIST:
    print(f'  MY{my}: {fail_pen[my]*100:.2f}%')

## 5. Mejora 7: Población en Riesgo (At-Risk Population)

Para cada mes, calcula cuántos vehículos están dentro del período de garantía.
Este es el **driver físico fundamental** de las fallas y codifica el conocimiento
de la flota cerrada directamente en los modelos ML.

In [ ]:
def compute_at_risk_series(periods, df_vol_local):
    """N° de vehículos en garantía para cada período dado."""
    result = np.zeros(len(periods))
    for _, row in df_vol_local.iterrows():
        sp = row['sales_period_est']
        we = row['warr_end']
        n  = row['n_prod']
        for i, p in enumerate(periods):
            if sp <= p < we:
                result[i] += n
    return result

future_months = pd.period_range('2026-05', HORIZON_END, freq='M')
future_dates  = future_months.to_timestamp()
ref_months_6  = pd.period_range('2025-11', '2026-04', freq='M')

print('Calculando at-risk histórico y futuro...')
at_risk_hist   = compute_at_risk_series(full_idx, df_vol)
at_risk_future = compute_at_risk_series(future_months, df_vol)

at_risk_hist_stable = at_risk_hist[STABLE_START:]
AR_MAX = max(at_risk_hist.max(), at_risk_future.max())

nz_idx = np.where(at_risk_future > 0)[0]
last_nz = str(future_months[nz_idx[-1]]) if len(nz_idx) > 0 else 'N/A'

print(f'At-risk histórico (estable Ene-2023 → Abr-2026):')
print(f'  Máximo  : {at_risk_hist_stable.max():,.0f} vehículos')
print(f'  Mínimo  : {at_risk_hist_stable.min():,.0f} vehículos (Abr-2026)')
print(f'\nAt-risk futuro (pronóstico):')
for i, label in [(0,'Mes 1  (May-2026)'),(11,'Mes 12 (Abr-2027)'),
                 (23,'Mes 24 (Abr-2028)'),(35,'Mes 36 (Abr-2029)')]:
    print(f'  {label}: {at_risk_future[i]:,.0f}')
print(f'  Último mes > 0  : {last_nz}  (flota cerrada en {PROD_CUTOFF})')

## 6. Mejora 8: Feature Engineering v3 — 22 Features

**Cambios vs v2 (33 features):**
- Lags: 12 → 6 · Rolling: ventanas {3,6} solo (elimina 9 y 12)
- Elimina: `trend_sq`, `roll_max_12`, `month_sin2/cos2`
- **Agrega:** `at_risk_norm`, `at_risk_delta` (física) · `fail_rate_lag1/roll3` (tasa normalizada)
- Ratio n/p: **0.82 → 1.23**

In [ ]:
FEAT_ORDER = (
    [f'lag_{k}' for k in range(1, N_LAGS+1)] +         # 6 lags
    ['diff_1','diff_2','seas_diff'] +                    # 3 diferencias
    ['roll_mean_3','roll_std_3','roll_mean_6','roll_std_6','roll_max_6','roll_min_6'] +  # 6 rolling
    ['at_risk_norm','at_risk_delta'] +                   # 2 dominio físico
    ['fail_rate_lag1','fail_rate_roll3'] +               # 2 tasas
    ['trend','month_sin','month_cos']                    # 3 temporales
)  # Total: 22 features


def make_features_v3(series, at_risk_arr, start_offset=STABLE_START, n_lags=N_LAGS):
    """Construye matriz de features para entrenamiento."""
    s  = pd.Series(series.astype(float))
    r  = pd.Series(at_risk_arr.astype(float))
    n  = len(s)
    feat = pd.DataFrame(index=range(n))

    for lag in range(1, n_lags + 1):
        feat[f'lag_{lag}']  = s.shift(lag).values
    feat['diff_1']          = (s.shift(1) - s.shift(2)).values
    feat['diff_2']          = (s.shift(1) - s.shift(3)).values
    feat['seas_diff']       = (s.shift(1) - s.shift(13)).values
    feat['roll_mean_3']     = s.shift(1).rolling(3).mean().values
    feat['roll_std_3']      = s.shift(1).rolling(3).std(ddof=1).fillna(0).values
    feat['roll_mean_6']     = s.shift(1).rolling(6).mean().values
    feat['roll_std_6']      = s.shift(1).rolling(6).std(ddof=1).fillna(0).values
    feat['roll_max_6']      = s.shift(1).rolling(6).max().values
    feat['roll_min_6']      = s.shift(1).rolling(6).min().values
    feat['at_risk_norm']    = (r / (AR_MAX + 1e-9)).shift(1).fillna(0).values
    feat['at_risk_delta']   = r.diff(1).shift(1).fillna(0).values / (AR_MAX + 1e-9)
    fail_rate               = s / (r + 1e-9)
    feat['fail_rate_lag1']  = fail_rate.shift(1).values
    feat['fail_rate_roll3'] = fail_rate.shift(1).rolling(3).mean().values
    feat['trend']           = np.arange(n) / n
    months = [(11 + start_offset + i) % 12 for i in range(n)]
    feat['month_sin']       = np.sin(2 * np.pi * np.array(months) / 12)
    feat['month_cos']       = np.cos(2 * np.pi * np.array(months) / 12)
    feat['target']          = s.values

    feat = feat.dropna()
    return feat[FEAT_ORDER].values, feat['target'].values, FEAT_ORDER


def build_feature_row(hist, hist_ar, ai, n_h, n_lags=N_LAGS):
    """
    Construye vector de features para pronóstico recursivo.
    DEBE producir exactamente el mismo vector que make_features_v3().
    """
    h   = np.array(hist, dtype=float)
    r_h = np.array(hist_ar, dtype=float)
    n   = len(h)
    rn  = len(r_h)

    lags = [h[-k] for k in range(1, n_lags + 1)]
    d1   = h[-1] - h[-2]  if n  >= 2  else 0.0
    d2   = h[-1] - h[-3]  if n  >= 3  else 0.0
    seas = h[-1] - h[-13] if n  >= 13 else 0.0

    rm3  = h[-3:].mean()          if n  >= 3 else h.mean()
    rs3  = h[-3:].std(ddof=1)     if n  >= 3 else 0.0
    rm6  = h[-6:].mean()          if n  >= 6 else h.mean()
    rs6  = h[-6:].std(ddof=1)     if n  >= 6 else 0.0
    rx6  = h[-6:].max()           if n  >= 6 else h.max()
    rn6  = h[-6:].min()           if n  >= 6 else h.min()

    at_prev  = float(r_h[-1])     if rn >= 1 else 0.0
    at_prev2 = float(r_h[-2])     if rn >= 2 else at_prev
    at_norm  = at_prev  / (AR_MAX + 1e-9)
    at_delta = (at_prev - at_prev2) / (AR_MAX + 1e-9)

    fr1 = h[-1] / (r_h[-1] + 1e-9) if rn >= 1 else 0.0
    if rn >= 3 and n >= 3:
        fr3 = np.mean(h[-3:] / (r_h[-3:] + 1e-9))
    else:
        fr3 = fr1

    rm_month = (11 + STABLE_START + ai) % 12
    return np.array([*lags, d1, d2, seas,
                     rm3, rs3, rm6, rs6, rx6, rn6,
                     at_norm, at_delta, fr1, fr3,
                     ai / n_h,
                     np.sin(2*np.pi*rm_month/12),
                     np.cos(2*np.pi*rm_month/12)]).reshape(1, -1)


X_w, y_w, FEAT_COLS = make_features_v3(ts_w_stable, at_risk_hist_stable)
X_p, y_p, _         = make_features_v3(ts_p_stable, at_risk_hist_stable)

print(f'X garantía shape : {X_w.shape}  ({len(FEAT_COLS)} features × {len(y_w)} muestras)')
print(f'Ratio n/p        : {len(y_w)}/{len(FEAT_COLS)} = {len(y_w)/len(FEAT_COLS):.2f}  (v2 era 0.82)')
print(f'\nFeatures:')
for i in range(0, len(FEAT_COLS), 4):
    print('  ' + '  '.join(f'[{i+j:02d}] {FEAT_COLS[i+j]:<20}'
                             for j in range(4) if i+j < len(FEAT_COLS)))

## 7. Mejora 9: XGBoost + Random Forest + ElasticNet + Ridge

**Por qué Random Forest?** Ensemble por naturaleza, validación OOB automática, menos overfitting que XGBoost con n pequeño.  
**Por qué ElasticNet?** L1+L2 regularización: selección automática de features y control de colinealidad.  
**XGBoost más conservador:** max_depth 4→3, n_estimators 400→200, min_child_weight 3→5.

In [ ]:
XGB_PARAMS = dict(
    n_estimators=200, learning_rate=0.05, max_depth=3,
    subsample=0.8, colsample_bytree=0.6, min_child_weight=5,
    reg_alpha=0.5, reg_lambda=3.0, gamma=0.3, random_state=42, verbosity=0
)
RF_PARAMS = dict(
    n_estimators=500, max_depth=6, min_samples_leaf=3, min_samples_split=5,
    max_features='sqrt', oob_score=True, random_state=42, n_jobs=-1
)
RIDGE_ALPHAS   = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 50.0, 100.0, 500.0]
ENET_L1_RATIOS = [0.1, 0.3, 0.5, 0.7, 0.9]
ENET_ALPHAS    = [0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]

scaler_w = StandardScaler(); X_w_sc = scaler_w.fit_transform(X_w)
scaler_p = StandardScaler(); X_p_sc = scaler_p.fit_transform(X_p)

xgb_w  = XGBRegressor(**XGB_PARAMS);  xgb_w.fit(X_w, y_w)
xgb_p  = XGBRegressor(**XGB_PARAMS);  xgb_p.fit(X_p, y_p)
rf_w   = RandomForestRegressor(**RF_PARAMS); rf_w.fit(X_w, y_w)
rf_p   = RandomForestRegressor(**RF_PARAMS); rf_p.fit(X_p, y_p)
enet_w = ElasticNetCV(l1_ratio=ENET_L1_RATIOS, alphas=ENET_ALPHAS,
                      cv=TimeSeriesSplit(4), max_iter=5000)
enet_w.fit(X_w_sc, y_w)
enet_p = ElasticNetCV(l1_ratio=ENET_L1_RATIOS, alphas=ENET_ALPHAS,
                      cv=TimeSeriesSplit(4), max_iter=5000)
enet_p.fit(X_p_sc, y_p)
ridge_w = RidgeCV(alphas=RIDGE_ALPHAS); ridge_w.fit(X_w_sc, y_w)
ridge_p = RidgeCV(alphas=RIDGE_ALPHAS); ridge_p.fit(X_p_sc, y_p)

enet_selected = int(np.sum(enet_w.coef_ != 0))
print(f'=== Modelos v3 entrenados ===')
print(f'  XGBoost  max_depth={XGB_PARAMS["max_depth"]} n_est={XGB_PARAMS["n_estimators"]} '
      f'min_child={XGB_PARAMS["min_child_weight"]}')
print(f'  RF OOB score garantía     : {rf_w.oob_score_:.4f} ({rf_w.oob_score_*100:.1f}%)')
print(f'  RF OOB score sin garantía : {rf_p.oob_score_:.4f}')
print(f'  ElasticNet α={enet_w.alpha_:.4f} l1_ratio={enet_w.l1_ratio_:.1f} '
      f'features activos: {enet_selected}/{len(FEAT_COLS)}')
print(f'  Ridge α óptimo: {ridge_w.alpha_}')

nz_w = y_w > 0
print(f'\nMAPE in-sample (informativo — NO se usa para pesos):')
for nm, pred in [
    ('XGBoost',     np.maximum(xgb_w.predict(X_w)[nz_w], 0)),
    ('RandomForest',np.maximum(rf_w.predict(X_w)[nz_w], 0)),
    ('ElasticNet',  np.maximum(enet_w.predict(X_w_sc)[nz_w], 0)),
    ('Ridge',       np.maximum(ridge_w.predict(X_w_sc)[nz_w], 0)),
]:
    mape = mean_absolute_percentage_error(y_w[nz_w]+1e-9, pred+1e-9)*100
    flag = '⚠️ probable overfitting' if mape < 2 else '✓'
    print(f'  {nm:<14}: {mape:.2f}%  {flag}')

## 8. Holt-Winters y SARIMA

In [ ]:
hw_w = ExponentialSmoothing(pd.Series(ts_w_stable).clip(lower=0.1),
                            trend='add', seasonal='add', seasonal_periods=12,
                            damped_trend=True).fit(optimized=True)
hw_fitted  = np.maximum(np.asarray(hw_w.fittedvalues).flatten(), 0)
mape_hw_is = mean_absolute_percentage_error(
    ts_w_stable[ts_w_stable>0]+1e-9, hw_fitted[ts_w_stable>0]+1e-9)*100

MAX_PRED = ts_w.max() * 10
best_aic = np.inf; bo = None; bso = None
print('Grid search SARIMA estable...')
for order in product([0,1,2],[0,1],[0,1,2]):
    for sorder in product([0,1],[0,1],[0,1]):
        try:
            r = SARIMAX(ts_w, order=order, seasonal_order=sorder+(12,),
                        enforce_stationarity=False, enforce_invertibility=False
                       ).fit(disp=False, maxiter=150)
            if not r.mle_retvals.get('converged', True): continue
            fc = np.asarray(r.get_forecast(12).predicted_mean).flatten()
            if not np.all(np.isfinite(fc)) or fc.max() > MAX_PRED: continue
            if r.aic < best_aic:
                best_aic = r.aic; bo = order; bso = sorder
        except: pass

sarima_w = SARIMAX(ts_w, order=bo, seasonal_order=bso+(12,),
                   enforce_stationarity=False, enforce_invertibility=False
                  ).fit(disp=False, maxiter=150)
print(f'SARIMA: {bo}×{bso+(12,)}  AIC={best_aic:.2f}')
sarima_fitted = np.maximum(np.asarray(sarima_w.fittedvalues).flatten(), 0)
mape_sar_is   = mean_absolute_percentage_error(
    ts_w[ts_w>0]+1e-9, sarima_fitted[ts_w>0]+1e-9)*100
print(f'HW in-sample MAPE    : {mape_hw_is:.2f}%')
print(f'SARIMA in-sample MAPE: {mape_sar_is:.2f}%')

## 9. Mejoras 1+10: CV-MAPE y Meta-Learner (Stacking)

**Mejora 1:** Pesos por MAPE de validación cruzada (no in-sample)  
**Mejora 10:** Meta-learner Ridge sobre OOF predictions — más riguroso que MAPE-inverso  
**Criterio SARIMA:** Si CV-MAPE > 15% → excluir del ensamble

In [ ]:
tscv       = TimeSeriesSplit(n_splits=6, gap=0, test_size=2)
model_keys = ['xgb','rf','enet','ridge','hw','sar']
cv_mapes   = {k: [] for k in model_keys}
n_train    = len(y_w)
oof_preds  = {k: np.zeros(n_train) for k in ['xgb','rf','enet','ridge']}

print('TSCV 6-fold — CV-MAPE y OOF predictions para stacking:')
for fold, (tr, val) in enumerate(tscv.split(X_w)):
    if len(tr) < 14: continue
    ts_tr = (ts_w_stable[:len(tr)+12] if len(tr)+12 <= len(ts_w_stable)
             else ts_w_stable[:len(tr)])

    m_x = XGBRegressor(**XGB_PARAMS);        m_x.fit(X_w[tr], y_w[tr])
    p_x = np.maximum(m_x.predict(X_w[val]), 0)

    m_rf2 = RandomForestRegressor(**RF_PARAMS); m_rf2.fit(X_w[tr], y_w[tr])
    p_rf2 = np.maximum(m_rf2.predict(X_w[val]), 0)

    sc_e = StandardScaler(); sc_e.fit(X_w[tr])
    m_en = ElasticNetCV(l1_ratio=ENET_L1_RATIOS, alphas=ENET_ALPHAS,
                        cv=TimeSeriesSplit(3), max_iter=3000)
    m_en.fit(sc_e.transform(X_w[tr]), y_w[tr])
    p_en = np.maximum(m_en.predict(sc_e.transform(X_w[val])), 0)

    m_rg = RidgeCV(alphas=RIDGE_ALPHAS)
    m_rg.fit(sc_e.transform(X_w[tr]), y_w[tr])
    p_rg = np.maximum(m_rg.predict(sc_e.transform(X_w[val])), 0)

    try:
        h_ = ExponentialSmoothing(pd.Series(ts_tr).clip(lower=0.1),
                                  trend='add', seasonal='add', seasonal_periods=12,
                                  damped_trend=True).fit(optimized=True)
        p_h = np.maximum(np.asarray(h_.forecast(len(val))), 0)
    except: p_h = p_x.copy()

    try:
        s_ = SARIMAX(ts_tr, order=bo, seasonal_order=bso+(12,),
                     enforce_stationarity=False, enforce_invertibility=False
                    ).fit(disp=False, maxiter=100)
        p_s = np.maximum(np.asarray(
            s_.get_forecast(len(val)).predicted_mean).flatten(), 0)
    except: p_s = p_x.copy()

    eps = 1e-9; nzv = y_w[val] > 0
    for k, pred in zip(model_keys, [p_x, p_rf2, p_en, p_rg, p_h, p_s]):
        if nzv.sum() > 0:
            cv_mapes[k].append(
                mean_absolute_percentage_error(y_w[val][nzv]+eps, pred[nzv]+eps)*100)

    oof_preds['xgb'][val]   = p_x
    oof_preds['rf'][val]    = p_rf2
    oof_preds['enet'][val]  = p_en
    oof_preds['ridge'][val] = p_rg

    print(f'  Fold {fold+1}: '
          + '  '.join(f'{k.upper()}={cv_mapes[k][-1]:5.1f}%'
                       for k in model_keys if cv_mapes[k]))

mape_cv = {k: np.mean(cv_mapes[k]) for k in model_keys if cv_mapes[k]}
mape_sd = {k: np.std(cv_mapes[k])  for k in model_keys if cv_mapes[k]}

INCLUDE_SARIMA = mape_cv.get('sar', 999) <= SARIMA_THRESHOLD
active_models  = [k for k in model_keys if k in mape_cv and (k!='sar' or INCLUDE_SARIMA)]

print(f'\nCV-MAPE promedios:')
for k in model_keys:
    if k in mape_cv:
        note = f'  ⚠️ EXCLUIDO (>{SARIMA_THRESHOLD}%)' if k=='sar' and not INCLUDE_SARIMA else ''
        print(f'  {k.upper():<8}: {mape_cv[k]:.2f}% ± {mape_sd[k]:.2f}%{note}')
print(f'  SARIMA incluido: {INCLUDE_SARIMA}')

# ── Pesos MAPE-inverso ────────────────────────────────────────────────────────
mapes_arr     = np.array([mape_cv[k] for k in active_models])
inv_mape      = 1.0 / mapes_arr
weights_mape  = dict(zip(active_models, inv_mape / inv_mape.sum()))

# ── Meta-learner Ridge (stacking) ─────────────────────────────────────────────
oof_ml_keys = ['xgb','rf','enet','ridge']
valid_mask  = np.any(np.column_stack([oof_preds[k] > 0 for k in oof_ml_keys]), axis=1)
USE_STACKING_FINAL = False
stack_weights = np.array([0.25, 0.25, 0.25, 0.25])

if valid_mask.sum() >= 10:
    oof_mat  = np.column_stack([oof_preds[k] for k in oof_ml_keys])[valid_mask]
    oof_y    = y_w[valid_mask]
    meta     = RidgeCV(alphas=[0.001,0.01,0.1,1.0,10.0,100.0], positive=True)
    meta.fit(oof_mat, oof_y)
    stack_weights     = meta.coef_ / meta.coef_.sum()
    nz_oof            = oof_y > 0
    mape_stack_oof    = mean_absolute_percentage_error(
        oof_y[nz_oof]+1e-9, (oof_mat @ stack_weights)[nz_oof]+1e-9)*100
    mape_mape_inv_est = 1.0 / np.sum(inv_mape / mapes_arr)
    USE_STACKING_FINAL = mape_stack_oof <= mape_mape_inv_est * 1.05
    print(f'\nMeta-learner OOF ({valid_mask.sum()} puntos válidos):')
    for k, w in zip(oof_ml_keys, stack_weights):
        print(f'  {k.upper():<8}: {w:.3f}')
    print(f'  MAPE stacking OOF : {mape_stack_oof:.2f}%')
    print(f'  MAPE MAPE-inverso : {mape_mape_inv_est:.2f}%')
    print(f'  → Usar stacking   : {USE_STACKING_FINAL}')

# ── Pesos finales ─────────────────────────────────────────────────────────────
if USE_STACKING_FINAL:
    w_xgb, w_rf, w_enet, w_ridge = stack_weights
else:
    w_xgb  = weights_mape.get('xgb',   0.25)
    w_rf   = weights_mape.get('rf',    0.25)
    w_enet = weights_mape.get('enet',  0.25)
    w_ridge= weights_mape.get('ridge', 0.25)

w_hw  = weights_mape.get('hw',  0.15)
w_sar = weights_mape.get('sar', 0.0) if INCLUDE_SARIMA else 0.0
total_w = w_xgb+w_rf+w_enet+w_ridge+w_hw+w_sar
w_xgb/=total_w; w_rf/=total_w; w_enet/=total_w
w_ridge/=total_w; w_hw/=total_w; w_sar/=total_w

mape_ensemble_cv = np.sum(np.array([mape_cv.get(k,10) for k in active_models]) *
                           np.array([weights_mape.get(k,0) for k in active_models]))
print(f'\n=== PESOS FINALES DEL ENSAMBLE ===')
for nm, w in [('XGBoost',w_xgb),('RF',w_rf),('ElasticNet',w_enet),
              ('Ridge',w_ridge),('HoltWinters',w_hw),('SARIMA',w_sar)]:
    print(f'  {nm:<12}: {w:.3f} ({w*100:.1f}%)')
print(f'\nMAPE CV estimado  : {mape_ensemble_cv:.2f}%')
print(f'Accuracy estimada : {100-mape_ensemble_cv:.2f}%')

## 10. Mejora 2: Bootstrap Residual para IC Genuinos

In [ ]:
min_train    = 16
residuos_abs = []
print('Walk-forward para residuos bootstrap:')
for t in range(min_train, len(y_w)):
    sc_ = StandardScaler(); sc_.fit(X_w[:t])
    m_x   = XGBRegressor(**XGB_PARAMS);          m_x.fit(X_w[:t], y_w[:t])
    m_rf2 = RandomForestRegressor(**RF_PARAMS);   m_rf2.fit(X_w[:t], y_w[:t])
    m_en  = ElasticNetCV(l1_ratio=ENET_L1_RATIOS, alphas=ENET_ALPHAS,
                         cv=TimeSeriesSplit(3), max_iter=2000)
    m_en.fit(sc_.transform(X_w[:t]), y_w[:t])
    m_rg  = RidgeCV(alphas=RIDGE_ALPHAS); m_rg.fit(sc_.transform(X_w[:t]), y_w[:t])

    p_x   = float(np.maximum(m_x.predict(X_w[t:t+1])[0],  0))
    p_rf2 = float(np.maximum(m_rf2.predict(X_w[t:t+1])[0], 0))
    p_en  = float(np.maximum(m_en.predict(sc_.transform(X_w[t:t+1]))[0], 0))
    p_rg  = float(np.maximum(m_rg.predict(sc_.transform(X_w[t:t+1]))[0], 0))

    ml_sum = w_xgb + w_rf + w_enet + w_ridge
    p_avg  = (w_xgb*p_x + w_rf*p_rf2 + w_enet*p_en + w_ridge*p_rg) / max(ml_sum, 1e-9)
    residuos_abs.append(y_w[t] - p_avg)

residuos_abs = np.array(residuos_abs)
print(f'  N residuos  : {len(residuos_abs)}')
print(f'  Media       : {residuos_abs.mean():+.1f}  (sesgo; idealmente ~0)')
print(f'  Std         : {residuos_abs.std():.1f}')
print(f'  Rango       : [{residuos_abs.min():+.1f}, {residuos_abs.max():+.1f}]')

np.random.seed(42)
ic_lo_boot = np.zeros(HORIZON)
ic_hi_boot = np.zeros(HORIZON)
typical_level = ts_w_stable.mean()
for h in range(HORIZON):
    factor = np.sqrt(h+1) if h <= 24 else np.sqrt(25) * (1 + np.log(h/24)/3)
    boot = np.random.choice(residuos_abs, size=N_BOOTSTRAP, replace=True) * factor
    ic_lo_boot[h] = np.percentile(boot, 5)
    ic_hi_boot[h] = np.percentile(boot, 95)
ic_lo_boot = np.maximum(ic_lo_boot, -typical_level*1.5)
ic_hi_boot = np.minimum(ic_hi_boot,  typical_level*1.5)

print(f'\nIC 90% bootstrap (fallas absolutas):')
for h in [0,5,11,23,59,119]:
    print(f'  Mes {h+1:>3}: [{ic_lo_boot[h]:+6.0f}, {ic_hi_boot[h]:+6.0f}]')

## 11. Índice de Actividad Weibull con Parámetros Corregidos

In [ ]:
def wb_hazard_inc(t, sh, sc):
    if t <= 0: return 0.0
    surv = max(1.0 - stats.weibull_min.cdf(max(t-1,0), sh, 0, sc), 1e-9)
    return stats.weibull_min.pdf(t, sh, 0, sc) / surv


def compute_weibull_forecast(df_vol_local, offset, wb_params_local=None):
    """Forecast Weibull escalado al nivel histórico reciente."""
    if wb_params_local is None:
        wb_params_local = wb_params
    dv = df_vol_local.copy()
    dv['sales_period_est'] = dv['prod_period'] + offset
    dv['warr_end']         = dv['sales_period_est'] + WARR_MONTHS

    idx_fut = pd.Series(0.0, index=future_months)
    idx_ref = pd.Series(0.0, index=ref_months_6)
    for _, row in dv.iterrows():
        my = int(row['model_year']); n = row['n_prod']
        sp = row['sales_period_est']; we = row['warr_end']
        if my not in wb_params_local: continue
        sh = wb_params_local[my]['sh']; sc = wb_params_local[my]['sc']
        for m in future_months:
            if m >= we: continue
            mis = max((m-sp).n, 0)
            if mis > 0: idx_fut[m] += n * wb_hazard_inc(mis, sh, sc)
        for m in ref_months_6:
            if m >= we: continue
            mis = max((m-sp).n, 0)
            if mis > 0: idx_ref[m] += n * wb_hazard_inc(mis, sh, sc)

    base  = ts_w_stable[-6:].mean()
    r_avg = idx_ref.mean()
    scale = base / r_avg if r_avg > 0 else 1.0
    return (idx_fut * scale).clip(lower=0).values


weib_fc_g_central = compute_weibull_forecast(df_vol, OFFSET_MEDIAN)
weib_fc_p_central = weib_fc_g_central * post_ratio
print(f'Weibull (params censurados, offset={OFFSET_MEDIAN}m):')
print(f'  Garantía total   : {weib_fc_g_central.sum():,.0f}')
print(f'  Sin garantía tot.: {weib_fc_p_central.sum():,.0f}')
print(f'  Pico mensual     : {weib_fc_g_central.max():.0f}/mes ({future_months[weib_fc_g_central.argmax()]})')
print(f'  Último mes > 0   : {future_months[np.where(weib_fc_g_central > 0.5)[0][-1]] if (weib_fc_g_central > 0.5).any() else "N/A"}')

## 12. Mejora 5: Monte Carlo del Offset Producción→Venta

In [ ]:
print(f'Monte Carlo con {N_MC} simulaciones del offset...')
np.random.seed(42)
mc_results = np.zeros((N_MC, HORIZON))

for sim in range(N_MC):
    dv_sim = df_vol.copy()
    offsets_sim = np.random.choice(OFFSET_DIST, size=len(dv_sim), replace=True)
    dv_sim['sales_period_est'] = [pp + int(off) for pp, off in
                                   zip(dv_sim['prod_period'], offsets_sim)]
    dv_sim['warr_end'] = dv_sim['sales_period_est'] + WARR_MONTHS

    idx_fut     = np.zeros(HORIZON)
    idx_ref_sum = 0.0
    for _, row in dv_sim.iterrows():
        my = int(row['model_year']); n = row['n_prod']
        sp = row['sales_period_est']; we = row['warr_end']
        if my not in wb_params: continue
        sh = wb_params[my]['sh']; sc = wb_params[my]['sc']
        for m in ref_months_6:
            if m >= we: continue
            mis = max((m-sp).n, 0)
            if mis > 0: idx_ref_sum += n * wb_hazard_inc(mis, sh, sc)
        for i, m in enumerate(future_months):
            if m >= we: continue
            mis = max((m-sp).n, 0)
            if mis > 0: idx_fut[i] += n * wb_hazard_inc(mis, sh, sc)

    base    = ts_w_stable[-6:].mean()
    ref_avg = idx_ref_sum / len(ref_months_6)
    scale   = base / ref_avg if ref_avg > 0 else 1.0
    mc_results[sim] = idx_fut * scale

weib_mc_median = np.median(mc_results, axis=0)
weib_mc_lo     = np.percentile(mc_results,  5, axis=0)
weib_mc_hi     = np.percentile(mc_results, 95, axis=0)
weib_fc_g      = weib_mc_median
weib_fc_p      = weib_fc_g * post_ratio

print(f'Resultados MC (Weibull censurado):')
print(f'  Total mediano : {weib_mc_median.sum():,.0f}')
print(f'  IC 90%        : [{weib_mc_lo.sum():,.0f} – {weib_mc_hi.sum():,.0f}]')
for h in [0,11,23]:
    print(f'  Mes {h+1:>2} [P5-P95]: [{weib_mc_lo[h]:.0f} – {weib_mc_hi[h]:.0f}]')

## 13. Mejora 3: Backtest Extendido 24 Meses

In [ ]:
CUTOFF_BT    = pd.Period('2024-04','M')
df_train_bt  = df[df['cal_month'] <= CUTOFF_BT].copy()
print(f'Train backtest: hasta {CUTOFF_BT} ({len(df_train_bt):,} fallas)')

wb_train_bt = {}
for my in MY_LIST:
    mis = df_train_bt[(df_train_bt['Model Year']==my) & df_train_bt['en_garantia']]['MIS'].dropna().values
    if len(mis) < 10: continue
    # Usar parámetros con censura también para el backtest
    sub_bt = df_vol[df_vol['model_year']==my].copy()
    sub_bt['sp_bt'] = sub_bt['prod_period'] + OFFSET_MEDIAN
    sub_bt['mis_at_cutoff'] = sub_bt['sp_bt'].apply(
        lambda s: max((CUTOFF_BT - s).n, 0))
    sub_bt['cens_t_bt'] = np.minimum(sub_bt['mis_at_cutoff'].values, WARR_MONTHS)
    tp = sub_bt['n_prod'].sum()
    n_park_bt = int(park_by_my.get(my, 0))
    n_cens_bt = max(n_park_bt - len(mis), 0)
    sub_bt['n_cbt'] = (sub_bt['n_prod']/tp*n_cens_bt).round().clip(lower=0).astype(int)
    cens_bt = sub_bt.groupby('cens_t_bt')['n_cbt'].sum().to_dict()
    sh_bt, sc_bt = fit_weibull_censored(mis, cens_bt)
    wb_train_bt[my] = {'sh': sh_bt, 'sc': sc_bt}

future_bt  = pd.period_range('2024-05','2026-04', freq='M')
ref_bt6    = pd.period_range('2023-11','2024-04', freq='M')

idx_fut_bt  = pd.Series(0.0, index=future_bt)
idx_ref_bt  = pd.Series(0.0, index=ref_bt6)
for _, row in df_vol.iterrows():
    my = int(row['model_year'])
    if my not in wb_train_bt: continue
    n  = row['n_prod']
    sp = row['prod_period'] + OFFSET_MEDIAN
    we = sp + WARR_MONTHS
    sh = wb_train_bt[my]['sh']; sc = wb_train_bt[my]['sc']
    for m in future_bt:
        if m >= we: continue
        mis = max((m-sp).n, 0)
        if mis > 0: idx_fut_bt[m] += n * wb_hazard_inc(mis, sh, sc)
    for m in ref_bt6:
        if m >= we: continue
        mis = max((m-sp).n, 0)
        if mis > 0: idx_ref_bt[m] += n * wb_hazard_inc(mis, sh, sc)

ts_to_cutoff = ts_w[:STABLE_START+16]
base_bt  = ts_to_cutoff[-6:].mean()
scale_bt = base_bt / idx_ref_bt.mean() if idx_ref_bt.mean() > 0 else 1.0
weib_bt  = (idx_fut_bt * scale_bt).clip(lower=0).values

ts_test_bt = ts_w[STABLE_START+16:STABLE_START+16+24]
nz_bt      = ts_test_bt > 0
mape_bt_12 = mean_absolute_percentage_error(
    ts_test_bt[:12][nz_bt[:12]]+1e-9, weib_bt[:12][nz_bt[:12]]+1e-9)*100
mape_bt_24 = mean_absolute_percentage_error(
    ts_test_bt[nz_bt]+1e-9, weib_bt[nz_bt]+1e-9)*100

print(f'\n=== BACKTEST 24 MESES — Weibull MLE censurado (Mejoras 3+6) ===')
print(f'  MAPE primeros 12m : {mape_bt_12:.1f}%  (Accuracy {100-mape_bt_12:.1f}%)')
print(f'  MAPE 24m completos: {mape_bt_24:.1f}%  (Accuracy {100-mape_bt_24:.1f}%)')
print(f'\n{"Mes":<4} {"Real":>6} {"Pred":>6} {"Err%":>6}')
for i in range(24):
    real = ts_test_bt[i]; pred = weib_bt[i]
    err  = abs(real-pred)/real*100 if real > 0 else 0
    if i < 4 or 10 <= i < 13 or 21 <= i:
        print(f'{i+1:<4} {real:>6.0f} {pred:>6.0f} {err:>5.1f}%')
    elif i == 4 or i == 13: print('  ...')

## 14. Mejora 11: Ensamble Final con Transición Weibull Anticipada

**Justificación:** Con la flota cerrada desde Nov-2024, el Weibull es el modelo físico
verdadero. El ML solo aporta valor en el corto plazo (suavizado y ajuste estacional).

- `XGB_SHORT=6`: Weibull empieza a entrar desde el **mes 7** (v2: mes 13)
- `XGB_BLEND=12`: ML se apaga completamente en el **mes 13** (v2: mes 25)

In [ ]:
def recursive_forecast_v3(model, ts_arr, at_risk_hist_arr, at_risk_fut_arr,
                           n_steps, n_lags=N_LAGS, is_linear=False, scaler=None):
    """Pronóstico recursivo con features de at-risk population (v3)."""
    clip_val = np.percentile(ts_arr[ts_arr>0], 99)*2.5 if ts_arr.max()>0 else 1000
    hist    = list(ts_arr.copy())
    hist_ar = list(at_risk_hist_arr.copy())
    n_h     = len(ts_arr)
    preds   = []
    for step in range(n_steps):
        ai  = n_h + step
        row = build_feature_row(hist, hist_ar, ai, n_h, n_lags)
        if is_linear: row = scaler.transform(row)
        pred = float(np.clip(model.predict(row)[0], 0, clip_val))
        preds.append(pred)
        hist.append(pred)
        r_fut = float(at_risk_fut_arr[step]) if step < len(at_risk_fut_arr) else 0.0
        hist_ar.append(r_fut)
    return np.array(preds)


# Pronósticos individuales (120 meses)
xgb_w_fc   = recursive_forecast_v3(xgb_w,   ts_w_stable, at_risk_hist_stable, at_risk_future, HORIZON)
xgb_p_fc   = recursive_forecast_v3(xgb_p,   ts_p_stable, at_risk_hist_stable, at_risk_future, HORIZON)
rf_w_fc    = recursive_forecast_v3(rf_w,    ts_w_stable, at_risk_hist_stable, at_risk_future, HORIZON)
rf_p_fc    = recursive_forecast_v3(rf_p,    ts_p_stable, at_risk_hist_stable, at_risk_future, HORIZON)
enet_w_fc  = recursive_forecast_v3(enet_w,  ts_w_stable, at_risk_hist_stable, at_risk_future, HORIZON,
                                    is_linear=True, scaler=scaler_w)
enet_p_fc  = recursive_forecast_v3(enet_p,  ts_p_stable, at_risk_hist_stable, at_risk_future, HORIZON,
                                    is_linear=True, scaler=scaler_p)
ridge_w_fc = recursive_forecast_v3(ridge_w, ts_w_stable, at_risk_hist_stable, at_risk_future, HORIZON,
                                    is_linear=True, scaler=scaler_w)
ridge_p_fc = recursive_forecast_v3(ridge_p, ts_p_stable, at_risk_hist_stable, at_risk_future, HORIZON,
                                    is_linear=True, scaler=scaler_p)
hw_w_fc     = np.maximum(hw_w.forecast(HORIZON).values, 0)
sarima_w_fc = np.maximum(np.asarray(
    sarima_w.get_forecast(HORIZON).predicted_mean).flatten(), 0)

# Ensamble ML ponderado
short_w = (w_xgb*xgb_w_fc + w_rf*rf_w_fc + w_enet*enet_w_fc + w_ridge*ridge_w_fc
           + w_hw*hw_w_fc + w_sar*sarima_w_fc)
ml_p_w  = w_xgb + w_rf + w_enet + w_ridge
short_p = ((w_xgb*xgb_p_fc + w_rf*rf_p_fc + w_enet*enet_p_fc + w_ridge*ridge_p_fc)
           / max(ml_p_w, 1e-9))

# Pesos de fase (Mejora 11: transición anticipada)
weights_xgb = np.array([
    1.0 if i < XGB_SHORT else
    1.0 - (i - XGB_SHORT) / (XGB_BLEND - XGB_SHORT) if i < XGB_BLEND else
    0.0 for i in range(HORIZON)])
weights_wb  = 1.0 - weights_xgb

hybrid_g = np.maximum(weights_xgb*short_w + weights_wb*weib_fc_g, 0)
hybrid_p = np.maximum(weights_xgb*short_p + weights_wb*weib_fc_p, 0)
hybrid_t = hybrid_g + hybrid_p

# IC final: combinación bootstrap + Monte Carlo
hybrid_lo = np.zeros(HORIZON)
hybrid_hi = np.zeros(HORIZON)
for h in range(HORIZON):
    long_lo  = weib_mc_lo[h] - weib_fc_g[h]
    long_hi  = weib_mc_hi[h] - weib_fc_g[h]
    delta_lo = weights_xgb[h]*ic_lo_boot[h] + weights_wb[h]*long_lo
    delta_hi = weights_xgb[h]*ic_hi_boot[h] + weights_wb[h]*long_hi
    hybrid_lo[h] = max(hybrid_t[h] + delta_lo, 0)
    hybrid_hi[h] = max(hybrid_t[h] + delta_hi, 0)

print('=== PRONÓSTICO v3 FINAL — 120 MESES ===')
print(f'  Modelos activos : XGB+RF+ENET+Ridge+HW' + ('+SAR' if INCLUDE_SARIMA else ''))
print(f'  Stacking usado  : {USE_STACKING_FINAL}')
print(f'  Trans. ML→Weib  : mes {XGB_SHORT} al {XGB_BLEND} (v2 era 12→24)')
print(f'  Garantía        : {hybrid_g.sum():>10,.0f} fallas')
print(f'  Sin garantía    : {hybrid_p.sum():>10,.0f} fallas')
print(f'  TOTAL           : {hybrid_t.sum():>10,.0f} fallas')
print(f'  IC 90%          : [{hybrid_lo.sum():>10,.0f} – {hybrid_hi.sum():>10,.0f}]')
print(f'  Pico mensual    : {hybrid_t.max():>10.0f}/mes ({future_months[hybrid_t.argmax()]})')

## 15. Proyección de Costos

In [ ]:
last6_w = df[df['en_garantia']  & (df['cal_month'] >= pd.Period('2025-11','M'))]
last6_p = df[~df['en_garantia'] & (df['cal_month'] >= pd.Period('2025-11','M'))]
cpf_w = last6_w['COSTS'].mean(); mat_w = last6_w['Material cost'].mean()
lab_w = last6_w['Labour cost'].mean()
cpf_p = last6_p['COSTS'].mean()          if len(last6_p) >= 5 else cpf_w
mat_p = last6_p['Material cost'].mean()  if len(last6_p) >= 5 else mat_w
lab_p = last6_p['Labour cost'].mean()    if len(last6_p) >= 5 else lab_w

infl_v   = (1.0 + INFLATION) ** (np.arange(1, HORIZON+1) / 12)
fc_tot_w = hybrid_g * cpf_w * infl_v
fc_tot_p = hybrid_p * cpf_p * infl_v
fc_mat_t = hybrid_g*mat_w*infl_v + hybrid_p*mat_p*infl_v
fc_lab_t = hybrid_g*lab_w*infl_v + hybrid_p*lab_p*infl_v
fc_tot_t = fc_tot_w + fc_tot_p
cpf_avg  = np.where(hybrid_t > 0,
                    (cpf_w*hybrid_g + cpf_p*hybrid_p)/np.maximum(hybrid_t,1e-9), cpf_w)
fc_cost_lo = np.nan_to_num(hybrid_lo * cpf_avg * infl_v, nan=0)
fc_cost_hi = np.nan_to_num(hybrid_hi * cpf_avg * infl_v, nan=0)

print(f'Costo base GARANTÍA  : ${cpf_w:,.2f}/falla  (Mat ${mat_w:.2f}  Lab ${lab_w:.2f})')
print(f'Costo base SIN GAR.  : ${cpf_p:,.2f}/falla  (Mat ${mat_p:.2f}  Lab ${lab_p:.2f})')
print(f'\n{"="*60}')
print(f'  PROYECCIÓN COSTOS — 120 MESES')
print(f'{"="*60}')
print(f'  Garantía     : ${fc_tot_w.sum():>12,.0f} USD')
print(f'  Sin garantía : ${fc_tot_p.sum():>12,.0f} USD')
print(f'  TOTAL        : ${fc_tot_t.sum():>12,.0f} USD')
print(f'  IC 90%       : [${fc_cost_lo.sum():,.0f} – ${fc_cost_hi.sum():,.0f}] USD')
print(f'{"="*60}')

## 16. Visualizaciones

In [ ]:
fig = plt.figure(figsize=(16, 22))
gs  = gridspec.GridSpec(4, 2, figure=fig, hspace=0.52, wspace=0.35)
ax1 = fig.add_subplot(gs[0,:])
ax2 = fig.add_subplot(gs[1,:])
ax3 = fig.add_subplot(gs[2,0])
ax4 = fig.add_subplot(gs[2,1])
ax5 = fig.add_subplot(gs[3,0])
ax6 = fig.add_subplot(gs[3,1])

# Panel 1: Pronóstico principal
ax1.fill_between(hist_dates, ts_w+ts_p, alpha=0.25, color=PALETTE[0])
ax1.plot(hist_dates, ts_w+ts_p, color=PALETTE[0], lw=2.5, label='Histórico')
ax1.fill_between(future_dates, hybrid_lo, hybrid_hi,
                 alpha=0.18, color=PALETTE[1], label='IC 90% (bootstrap+MC Weibull)')
ax1.plot(future_dates, hybrid_t, color=PALETTE[1], lw=2.5, ls='--',
         label=f'Pronóstico v3 ({hybrid_t.sum():,.0f} fallas)')
ax1.axvline(pd.Timestamp('2026-05-01'), color='black', ls='--', lw=1.5, alpha=0.6)
ax1.axvspan(future_dates[0], future_dates[min(XGB_SHORT-1,len(future_dates)-1)],
            alpha=0.07, color='green',  label=f'ML puro (m1-{XGB_SHORT})')
if XGB_BLEND <= len(future_dates):
    ax1.axvspan(future_dates[XGB_SHORT], future_dates[min(XGB_BLEND-1,len(future_dates)-1)],
                alpha=0.07, color='orange', label=f'Transición (m{XGB_SHORT+1}-{XGB_BLEND})')
    ax1.axvspan(future_dates[XGB_BLEND], future_dates[-1],
                alpha=0.07, color='royalblue', label='Weibull puro (m13+)')
ax1.set_title('Pronóstico v3 ROBUSTO — Oil Separator · IC bootstrap+MC Weibull censurado',
              fontsize=12, fontweight='bold')
ax1.set_ylabel('N° Fallas / Mes')
ax1.legend(fontsize=8, ncol=3)

# Panel 2: CV-MAPE por modelo
models_plot = [k for k in model_keys if k in mape_cv]
labels_p2   = [k.upper() for k in models_plot]
mapes_p2    = [mape_cv[k] for k in models_plot]
colors_p2   = [PALETTE[i] for i in range(len(models_plot))]
bars = ax2.bar(labels_p2, mapes_p2, color=colors_p2, alpha=0.85)
ax2.set_title('CV-MAPE por Modelo (Mejoras 1+9+10) — base para pesos del ensamble',
              fontweight='bold')
ax2.set_ylabel('MAPE %')
ax2.axhline(SARIMA_THRESHOLD, color='red', ls='--', lw=1,
            label=f'Umbral exclusión SARIMA ({SARIMA_THRESHOLD}%)')
for bar, v in zip(bars, mapes_p2):
    ax2.text(bar.get_x()+bar.get_width()/2, v+0.3, f'{v:.1f}%', ha='center', fontsize=9)
ax2b = ax2.twinx()
all_names = ['XGBoost','RF','ENET','Ridge','HoltWinters','SARIMA']
all_weights = [w_xgb, w_rf, w_enet, w_ridge, w_hw, w_sar]
final_w_plot = [all_weights[['xgb','rf','enet','ridge','hw','sar'].index(k)]
                for k in models_plot]
ax2b.plot(labels_p2, [w*100 for w in final_w_plot], 'ko-', markersize=10)
ax2b.set_ylabel('Peso final (%)')
ax2.legend(loc='upper left', fontsize=8)

# Panel 3: At-risk histórico y futuro (Mejora 7)
ax3.plot(hist_dates, at_risk_hist/1000, color=PALETTE[0], lw=2, label='Histórico')
ax3.plot(future_dates, at_risk_future/1000, color=PALETTE[1], lw=2, ls='--', label='Futuro')
ax3.axvline(pd.Timestamp('2024-11-01'), color='red', ls=':', lw=1.5,
            label='Fin producción (Nov-2024)')
ax3.set_title('At-Risk Population (Mejora 7) — Driver físico de fallas', fontweight='bold')
ax3.set_ylabel('Miles de vehículos en garantía')
ax3.legend(fontsize=8)

# Panel 4: Weibull β y η — prev vs corregido (Mejora 6)
x_my = np.arange(len(MY_LIST))
w4 = 0.35
beta_prev = [stats.weibull_min.fit(
    df[(df['Model Year']==my)&df['en_garantia']]['MIS'].dropna().values, floc=0)[0]
    for my in MY_LIST]
beta_corr = [wb_params[my]['sh'] for my in MY_LIST]
ax4.bar(x_my - w4/2, beta_prev, w4, label='β previo (sin censura)', color=PALETTE[2], alpha=0.8)
ax4.bar(x_my + w4/2, beta_corr, w4, label='β corregido (con censura)', color=PALETTE[3], alpha=0.8)
ax4.set_xticks(x_my); ax4.set_xticklabels([f'MY{m}' for m in MY_LIST])
ax4.set_title('Weibull β: parámetros previos vs corregidos (Mejora 6)', fontweight='bold')
ax4.set_ylabel('Forma β (Weibull)')
ax4.legend(fontsize=8)

# Panel 5: Bootstrap residuos
ax5.hist(residuos_abs, bins=max(len(residuos_abs)//2,5), color=PALETTE[4], alpha=0.75,
         edgecolor='black')
ax5.axvline(0, color='red', ls='--', lw=1.5, label='Sesgo=0')
ax5.axvline(residuos_abs.mean(), color='orange', ls='-', lw=1.5,
            label=f'Media={residuos_abs.mean():+.1f}')
ax5.set_title(f'Residuos Walk-Forward ML (n={len(residuos_abs)}) — Bootstrap IC',
              fontweight='bold')
ax5.set_xlabel('Residuo (fallas)'); ax5.set_ylabel('Frecuencia')
ax5.legend(fontsize=8)

# Panel 6: Comparación ensamble ML vs Weibull censurado
ax6.plot(future_dates[:36], short_w[:36],  color=PALETTE[6], lw=1.8, ls=':', label='Ensamble ML')
ax6.plot(future_dates[:36], weib_fc_g[:36], color=PALETTE[5], lw=1.8, ls='--', label='Weibull censurado')
ax6.plot(future_dates[:36], hybrid_g[:36],  color=PALETTE[1], lw=2.5, label='Híbrido (ponderado)')
ax6.axvline(future_dates[XGB_SHORT], color='gray', ls=':', lw=1, alpha=0.7)
ax6.axvline(future_dates[XGB_BLEND], color='gray', ls='-.', lw=1, alpha=0.7)
ax6.set_title('Comparación: ML vs Weibull vs Híbrido (36 meses)', fontweight='bold')
ax6.set_ylabel('Fallas garantía/mes'); ax6.legend(fontsize=8)

fig.suptitle('Pronóstico Robusto v3 — 11 Mejoras Metodológicas (6 nuevas en v3)',
             fontsize=14, fontweight='bold', y=1.002)
plt.savefig('fig_robust_forecast_v3.png', bbox_inches='tight')
plt.show()
print('✅ fig_robust_forecast_v3.png guardado')

## 17. Resumen Anual y Exportación

In [ ]:
forecast_table = pd.DataFrame({
    'Periodo'           : [str(p) for p in future_months],
    'Fase'              : ['ML' if i<XGB_SHORT else 'Transicion' if i<XGB_BLEND
                           else 'Weibull' for i in range(HORIZON)],
    'Garantia_Fallas'  : np.round(hybrid_g).astype(int),
    'SinGarantia_Fallas': np.round(hybrid_p).astype(int),
    'Total_Fallas'     : np.round(hybrid_t).astype(int),
    'IC_90_Inferior'   : np.round(hybrid_lo).astype(int),
    'IC_90_Superior'   : np.round(hybrid_hi).astype(int),
    'Mat_Cost_USD'     : np.round(fc_mat_t, 2),
    'Labour_Cost_USD'  : np.round(fc_lab_t, 2),
    'Total_Cost_USD'   : np.round(fc_tot_t, 2),
    'AtRisk_Vehiculos' : np.round(at_risk_future).astype(int),
})
forecast_table['Fallas_Acum']    = forecast_table['Total_Fallas'].cumsum()
forecast_table['Costo_Acum_USD'] = forecast_table['Total_Cost_USD'].cumsum().round(2)
forecast_table['Año']            = [str(p)[:4] for p in future_months]

annual = forecast_table.groupby('Año').agg(
    Garantia=('Garantia_Fallas','sum'), SinGarantia=('SinGarantia_Fallas','sum'),
    Total=('Total_Fallas','sum'), IC_Inf=('IC_90_Inferior','sum'),
    IC_Sup=('IC_90_Superior','sum'), Mat_Cost=('Mat_Cost_USD','sum'),
    Lab_Cost=('Labour_Cost_USD','sum'), Total_Cost=('Total_Cost_USD','sum'),
).reset_index()
annual['Fallas_Acum'] = annual['Total'].cumsum()
annual['Costo_Acum']  = annual['Total_Cost'].cumsum().round(0).astype(int)

print('RESUMEN ANUAL v3 CON IC ROBUSTOS')
print('='*115)
print(annual[['Año','Garantia','SinGarantia','Total','IC_Inf','IC_Sup',
              'Mat_Cost','Lab_Cost','Total_Cost','Fallas_Acum']].to_string(index=False))

validation_metrics = pd.DataFrame({
    'Metrica': ['CV-MAPE XGBoost','CV-MAPE RandomForest','CV-MAPE ElasticNet',
                'CV-MAPE Ridge','CV-MAPE HoltWinters','CV-MAPE SARIMA',
                'MAPE Ensamble (estimado)','SARIMA incluido',
                'Backtest 12m (Weibull censurado)','Backtest 24m (Weibull censurado)',
                'Stacking meta-learner usado'],
    'Valor'  : [round(mape_cv.get('xgb',0),2), round(mape_cv.get('rf',0),2),
                round(mape_cv.get('enet',0),2), round(mape_cv.get('ridge',0),2),
                round(mape_cv.get('hw',0),2),   round(mape_cv.get('sar',0),2),
                round(mape_ensemble_cv,2),       str(INCLUDE_SARIMA),
                round(mape_bt_12,2),             round(mape_bt_24,2),
                str(USE_STACKING_FINAL)],
})

output_file = 'Pronostico_v3_OilSeparator.xlsx'
with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
    forecast_table.drop(columns=['Año']).to_excel(
        writer, sheet_name='Pronostico_Mensual', index=False)
    annual.to_excel(writer, sheet_name='Resumen_Anual', index=False)
    validation_metrics.to_excel(writer, sheet_name='Metricas_Validacion', index=False)
    pd.DataFrame({
        'Periodo': full_idx.astype(str),
        'Garantia': hist_w.values.astype(int),
        'SinGarantia': hist_p.values.astype(int),
        'Total': hist_t.values.astype(int),
        'AtRisk': at_risk_hist.astype(int),
    }).to_excel(writer, sheet_name='Historico_Mensual', index=False)

    wb_df = pd.DataFrame([{
        'Model_Year': my,
        'Parque': int(park_by_my.get(my,0)),
        'Fallas_obs': int(fails_by_my.get(my,0)),
        'Fallas_en_garantia': len(df[(df['Model Year']==my)&df['en_garantia']]),
        'Penetracion_pct': round(fail_pen[my]*100, 3),
        'beta_corr': round(wb_params[my]['sh'], 4),
        'eta_corr': round(wb_params[my]['sc'], 2),
        'B50_meses': round(wb_params[my]['B50'], 1),
    } for my in MY_LIST])
    wb_df.to_excel(writer, sheet_name='Weibull_Censurado', index=False)

print(f'\n✅ Archivo guardado: {output_file}')

## 18. Conclusiones del Modelo v3

### Mejoras técnicas implementadas sobre v2

| Mejora | Impacto esperado |
|--------|------------------|
| **Weibull MLE censurado** | β y η corregidos para MY2023/2024 → mejor timing del pico de fallas |
| **At-risk como feature** | ML conoce la física de la flota cerrada → no extrapola hacia arriba |
| **Features 33→22** | Ratio n/p 0.82→1.23 → reduce overfitting estructural |
| **RF + ElasticNet** | OOB validation automática · selección L1 de features |
| **Stacking meta-learner** | Pesos basados en OOF reales (no heurísticos) |
| **Transición mes 7-12** | Weibull domina antes → menos dependencia de ML recursivo |

### Limitaciones estructurales (no resolubles sin más datos)
- **N_bootstrap residuos pequeño** → IC dependen de pocos puntos walk-forward
- **27 muestras estables** → modelos ML requieren regularización fuerte; RF es más robusto
- **Right-censoring residual MY2024** → penetración final puede subestimarse 10-20%
- **Para decisiones operacionales** → reentrenar cada 3 meses; monitorear drift real vs predicho
- **Para decisiones financieras** → usar el rango IC, no el valor central